In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# Environment & Setup

import os, sys, json, random, csv, zipfile
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, cohen_kappa_score
import matplotlib
matplotlib.use('Agg')

os.system('pip install -q timm pytorch-ignite')
from ignite.engine import Engine, Events
from ignite.metrics import Accuracy, Loss, RunningAverage
from ignite.handlers import EarlyStopping

# ── Paths & Setup ──────────────────────────────────────────────────────────
FOOD101_ROOT = '/kaggle/input/datasets/srujanesanakarra/food101/food-101'
IMAGES_DIR   = f'{FOOD101_ROOT}/images'
META_DIR     = f'{FOOD101_ROOT}/meta'

os.chdir('/kaggle/working')
if not os.path.exists('fyp-food-classification'):
    os.system('git clone https://github.com/Ahmad-techs/fyp-food-classification.git')
else:
    os.system('cd fyp-food-classification && git pull')
sys.path.insert(0, '/kaggle/working/fyp-food-classification/src')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device}')

def set_seeds(seed=42):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seeds(42)

OUT_DIR  = '/kaggle/working/results_vit'
CKPT_DIR = '/kaggle/working/checkpoints_vit'
ERR_DIR  = f'{OUT_DIR}/error_analysis'
os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(ERR_DIR,  exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print("Environment, paths, and seeds initialized successfully!\n")

Cloning into 'fyp-food-classification'...


Device configured: cuda
Environment, paths, and seeds initialized successfully!



In [4]:
# Taxonomy & Coarse Mapping

with open(f'{META_DIR}/classes.txt', 'r') as f:
    CLASS_NAMES = [line.strip() for line in f.readlines()]
NUM_CLASSES = len(CLASS_NAMES)

COARSE_GROUPS = {
    0: 'Desserts & Sweets', 1: 'Meat & Poultry', 2: 'Seafood',
    3: 'Pizza & Pasta', 4: 'Soups & Stews', 5: 'Salads & Vegetables',
    6: 'Fast Food & Sandwiches', 7: 'Breakfast & Eggs', 8: 'Asian & International',
}
NUM_COARSE = len(COARSE_GROUPS)

COARSE_MAP = {
    'apple_pie': 0, 'baklava': 0, 'beignets': 0, 'bread_pudding': 0, 'cannoli': 0,
    'carrot_cake': 0, 'cheesecake': 0, 'chocolate_cake': 0, 'chocolate_mousse': 0,
    'churros': 0, 'creme_brulee': 0, 'cup_cakes': 0, 'donuts': 0, 'frozen_yogurt': 0,
    'ice_cream': 0, 'macarons': 0, 'panna_cotta': 0, 'red_velvet_cake': 0,
    'strawberry_shortcake': 0, 'tiramisu': 0,
    'baby_back_ribs': 1, 'beef_carpaccio': 1, 'beef_tartare': 1, 'chicken_curry': 1,
    'chicken_wings': 1, 'filet_mignon': 1, 'foie_gras': 1, 'peking_duck': 1,
    'pork_chop': 1, 'prime_rib': 1, 'steak': 1,
    'ceviche': 2, 'crab_cakes': 2, 'fish_and_chips': 2, 'fried_calamari': 2,
    'grilled_salmon': 2, 'lobster_bisque': 2, 'lobster_roll_sandwich': 2,
    'mussels': 2, 'oysters': 2, 'sashimi': 2, 'scallops': 2,
    'shrimp_and_grits': 2, 'tuna_tartare': 2,
    'cheese_pizza': 3, 'gnocchi': 3, 'lasagna': 3, 'macaroni_and_cheese': 3,
    'pizza': 3, 'ravioli': 3, 'risotto': 3, 'spaghetti_bolognese': 3,
    'spaghetti_carbonara': 3,
    'clam_chowder': 4, 'french_onion_soup': 4, 'hot_and_sour_soup': 4,
    'miso_soup': 4, 'pho': 4, 'ramen': 4,
    'beet_salad': 5, 'caesar_salad': 5, 'caprese_salad': 5, 'edamame': 5,
    'greek_salad': 5, 'guacamole': 5, 'hummus': 5, 'seaweed_salad': 5,
    'breakfast_burrito': 6, 'bruschetta': 6, 'club_sandwich': 6,
    'croque_madame': 6, 'french_fries': 6, 'garlic_bread': 6,
    'grilled_cheese_sandwich': 6, 'hamburger': 6, 'hot_dog': 6, 'nachos': 6,
    'onion_rings': 6, 'pulled_pork_sandwich': 6, 'tacos': 6,
    'deviled_eggs': 7, 'eggs_benedict': 7, 'french_toast': 7,
    'huevos_rancheros': 7, 'pancakes': 7, 'waffles': 7,
    'bibimbap': 8, 'dumplings': 8, 'escargots': 8, 'falafel': 8,
    'fried_rice': 8, 'pad_thai': 8, 'paella': 8, 'poutine': 8, 'samosa': 8,
    'spring_rolls': 8, 'sushi': 8, 'takoyaki': 8,
}

def get_coarse_from_idx(fine_idx):
    return COARSE_MAP.get(CLASS_NAMES[fine_idx], 8)

print(f' Loaded {NUM_CLASSES} fine-grained classes and mapped to {NUM_COARSE} coarse categories.\n')

 Loaded 101 fine-grained classes and mapped to 9 coarse categories.



In [5]:
# Transforms & DataLoaders

def get_transforms(split):
    if split == 'train':
        return transforms.Compose([
            transforms.Resize(256), transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        ])
    return transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    ])

def load_split_paths(txt_file):
    paths, labels = [], []
    with open(txt_file, 'r') as f:
        for line in f:
            p = line.strip()
            paths.append(f"{p}.jpg")
            labels.append(CLASS_NAMES.index(p.split('/')[0]))
    return paths, labels

class Food101DiskDataset(Dataset):
    def __init__(self, data_list, split_name):
        self.data = data_list
        self.transform = get_transforms(split_name)

    def __len__(self): return len(self.data)

    def __getitem__(self, i):
        rel_path, fine = self.data[i]
        img = Image.open(os.path.join(IMAGES_DIR, rel_path)).convert('RGB')
        return self.transform(img), fine, get_coarse_from_idx(fine)

    def filename_at(self, i): return self.data[i][0]

train_paths, train_labels = load_split_paths(f'{META_DIR}/train.txt')
test_paths, test_labels   = load_split_paths(f'{META_DIR}/test.txt')

set_seeds(42)
test_combined = list(zip(test_paths, test_labels))
random.shuffle(test_combined)
half = len(test_combined) // 2
val_data, test_data = test_combined[:half], test_combined[half:]
train_data = list(zip(train_paths, train_labels))

BS = 32
train_ds = Food101DiskDataset(train_data, 'train')
val_ds   = Food101DiskDataset(val_data,   'val')
test_ds  = Food101DiskDataset(test_data,  'test')

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

print(f'Data loaders ready! Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}\n')

Data loaders ready! Train: 75750 | Val: 12625 | Test: 12625



In [6]:
# ViT Model Architecture

class ViTDualHead(nn.Module):
    def __init__(self, num_fine=NUM_CLASSES, num_coarse=NUM_COARSE, dropout=0.3):
        super().__init__()
        vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        vit.heads = nn.Identity()
        self.backbone = vit
        self.dropout   = nn.Dropout(dropout)
        self.fine_head   = nn.Linear(768, num_fine)
        self.coarse_head = nn.Linear(768, num_coarse)

    def forward(self, x):
        x = self.backbone(x); x = self.dropout(x)
        return self.fine_head(x), self.coarse_head(x)

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_top(self, n=3):
        layers = list(self.backbone.encoder.layers.children())
        for layer in layers[-n:]:
            for p in layer.parameters(): p.requires_grad = True
        for p in self.backbone.encoder.ln.parameters(): p.requires_grad = True

print(" ViTDualHead model class defined.\n")

 ViTDualHead model class defined.



In [7]:
# PyTorch Ignite Engines

LAM = 0.35

def make_loss_fn(lam):
    crit = nn.CrossEntropyLoss()
    return lambda fo, co, fl, cl: ((1 - lam) * crit(fo, fl) + lam * crit(co, cl)) if lam > 0 else crit(fo, fl)

def build_engines(model, lr, lam):
    loss_fn = make_loss_fn(lam)
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)

    def train_step(engine, batch):
        model.train()
        imgs, fl, cl = [t.to(device) if torch.is_tensor(t) else torch.tensor(t).to(device) for t in batch]
        optimizer.zero_grad()
        fo, co = model(imgs); loss = loss_fn(fo, co, fl, cl)
        loss.backward(); optimizer.step()
        return {'loss': loss.item()}

    def eval_step(engine, batch):
        model.eval()
        with torch.no_grad():
            imgs, fl, cl = [t.to(device) if torch.is_tensor(t) else torch.tensor(t).to(device) for t in batch]
            fo, co = model(imgs); loss = loss_fn(fo, co, fl, cl)
        return {'loss': loss.item(), 'fine_pred': fo, 'fine_y': fl, 'coarse_pred': co, 'coarse_y': cl}

    trainer, evaluator = Engine(train_step), Engine(eval_step)
    RunningAverage(output_transform=lambda o: o['loss']).attach(trainer, 'loss')
    Accuracy(output_transform=lambda o: (o['fine_pred'], o['fine_y'])).attach(evaluator, 'fine_acc')
    Accuracy(output_transform=lambda o: (o['coarse_pred'], o['coarse_y'])).attach(evaluator, 'coarse_acc')
    Loss(nn.CrossEntropyLoss(), output_transform=lambda o: (o['fine_pred'], o['fine_y'])).attach(evaluator, 'val_loss')
    return trainer, evaluator

def run_phase(model, tag, max_epochs, lr, lam, save_path, history, use_es, patience=5):
    set_seeds(42)
    trainer, evaluator = build_engines(model, lr, lam)
    best = {'fine_acc': history.get('best_fine_acc', 0.0)}

    @trainer.on(Events.EPOCH_COMPLETED)
    def _val(engine):
        evaluator.run(val_loader)
        m = evaluator.state.metrics
        fa, ca, vl = m['fine_acc']*100, m['coarse_acc']*100, m['val_loss']
        avg_loss = trainer.state.metrics.get('loss', float('nan'))
        print(f'  {tag} Ep{engine.state.epoch:03d}/{max_epochs} train_loss={avg_loss:.4f} val_loss={vl:.4f} Fine={fa:.2f}% Coarse={ca:.2f}%')
        history.setdefault('val_fine', []).append(fa)
        history.setdefault('val_coarse', []).append(ca)
        history.setdefault('val_loss', []).append(vl)
        history.setdefault('loss', []).append(avg_loss)
        if fa > best['fine_acc']:
            best['fine_acc'] = fa
            history['best_fine_acc'] = fa
            torch.save({'model_state_dict': model.state_dict(), 'best_fine_acc': fa,
                        'lambda': lam, 'epoch': engine.state.epoch, 'phase': tag}, save_path)
            print(f'    ✓ Checkpoint saved (best Fine={fa:.2f}%)')

    if use_es:
        es = EarlyStopping(patience=patience, score_function=lambda e: -e.state.metrics['val_loss'], trainer=trainer)
        evaluator.add_event_handler(Events.COMPLETED, es)

    trainer.run(train_loader, max_epochs=max_epochs)
    return history

print(f'Ignite engines and helper functions configured (lambda={LAM}).\n')

Ignite engines and helper functions configured (lambda=0.35).



In [8]:
# Execute ViT Training

ckpt_path = f'{CKPT_DIR}/food101_v3_vit.pth'
model = ViTDualHead().to(device)
h = {}

print('--> Phase 1: Training classifier heads with frozen backbone...')
model.freeze_backbone()
h = run_phase(model, 'P1', 10, 1e-3, LAM, ckpt_path, h, use_es=False)

print('\n--> Phase 2: Fine-tuning top backbone layers...')
model.unfreeze_top(n=3)
h = run_phase(model, 'P2', 100, 1e-4, LAM, ckpt_path, h, use_es=True, patience=5)

with open(f'{OUT_DIR}/vit_history.json', 'w') as f:
    json.dump({k: v for k, v in h.items() if isinstance(v, list)}, f)

print('\n Training complete! History logged to vit_history.json.\n')

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 214MB/s] 


--> Phase 1: Training classifier heads with frozen backbone...
  P1 Ep001/10 train_loss=1.3432 val_loss=1.0812 Fine=70.69% Coarse=70.04%
    ✓ Checkpoint saved (best Fine=70.69%)
  P1 Ep002/10 train_loss=1.2982 val_loss=0.9752 Fine=73.68% Coarse=70.86%
    ✓ Checkpoint saved (best Fine=73.68%)
  P1 Ep003/10 train_loss=1.3109 val_loss=0.9512 Fine=74.00% Coarse=71.14%
    ✓ Checkpoint saved (best Fine=74.00%)
  P1 Ep004/10 train_loss=1.3268 val_loss=0.9281 Fine=74.63% Coarse=70.89%
    ✓ Checkpoint saved (best Fine=74.63%)
  P1 Ep005/10 train_loss=1.3056 val_loss=0.9219 Fine=74.55% Coarse=70.46%
  P1 Ep006/10 train_loss=1.2749 val_loss=0.9151 Fine=74.70% Coarse=71.19%
    ✓ Checkpoint saved (best Fine=74.70%)
  P1 Ep007/10 train_loss=1.2227 val_loss=0.9028 Fine=74.78% Coarse=71.33%
    ✓ Checkpoint saved (best Fine=74.78%)
  P1 Ep008/10 train_loss=1.2472 val_loss=0.8953 Fine=75.61% Coarse=70.91%
    ✓ Checkpoint saved (best Fine=75.61%)
  P1 Ep009/10 train_loss=1.2487 val_loss=0.8843 Fin

2026-08-01 11:38:49,166 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


  P2 Ep008/100 train_loss=0.1583 val_loss=0.7366 Fine=84.10% Coarse=88.90%

 Training complete! History logged to vit_history.json.



In [9]:
# Evaluation & Predictions Export

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_ft, all_fp, all_ct, all_cp, all_files, all_conf = [], [], [], [], [], []
idx = 0
with torch.no_grad():
    for imgs, fl, cl in test_loader:
        fo, co = model(imgs.to(device))
        probs = torch.softmax(fo, dim=1)
        preds = fo.argmax(1).cpu().numpy()
        confs = probs.max(1).values.cpu().numpy()
        for b in range(imgs.size(0)):
            all_files.append(test_loader.dataset.filename_at(idx)); idx += 1
        all_fp.extend(preds); all_ft.extend(fl.numpy())
        all_cp.extend(co.argmax(1).cpu().numpy()); all_ct.extend(cl.numpy())
        all_conf.extend(confs)

all_ft, all_fp = np.array(all_ft), np.array(all_fp)
all_ct, all_cp = np.array(all_ct), np.array(all_cp)

acc = accuracy_score(all_ft, all_fp) * 100
f1 = f1_score(all_ft, all_fp, average='macro', zero_division=0) * 100
kappa = cohen_kappa_score(all_ft, all_fp)
coarse_acc = accuracy_score(all_ct, all_cp) * 100

print(f'\n[ViT-B/16 TEST RESULTS]')
print(f'Top-1 Fine Accuracy: {acc:.2f}%')
print(f'Macro F1 Score:       {f1:.2f}%')
print(f'Cohen Kappa:          {kappa:.4f}')
print(f'Coarse Top-1 Acc:     {coarse_acc:.2f}%\n')

pred_log_path = f'{OUT_DIR}/ViT-B_16_test_predictions.csv'
with open(pred_log_path, 'w', newline='') as fcsv:
    writer = csv.writer(fcsv)
    writer.writerow(['sample_id', 'true_label', 'pred_label', 'true_class_name', 'pred_class_name', 'confidence'])
    for fn, t, p, c in zip(all_files, all_ft, all_fp, all_conf):
        writer.writerow([fn, int(t), int(p), CLASS_NAMES[int(t)], CLASS_NAMES[int(p)], round(float(c), 4)])

print(f' Predictions saved to: {pred_log_path}\n')


[ViT-B/16 TEST RESULTS]
Top-1 Fine Accuracy: 84.39%
Macro F1 Score:       84.39%
Cohen Kappa:          0.8423
Coarse Top-1 Acc:     88.82%

 Predictions saved to: /kaggle/working/results_vit/ViT-B_16_test_predictions.csv



In [10]:
# Error Analysis

df = pd.read_csv(pred_log_path)
mis = df[df['true_label'] != df['pred_label']]
pair_counts = mis.groupby(['true_class_name', 'pred_class_name']).size().reset_index(name='count').sort_values('count', ascending=False)
pair_counts.to_csv(f'{ERR_DIR}/vit_confused_pairs.csv', index=False)

if len(pair_counts) > 0:
    top_true, top_pred = pair_counts.iloc[0]['true_class_name'], pair_counts.iloc[0]['pred_class_name']
    print(f'Most common confusion: "{top_true}" misclassified as "{top_pred}" ({pair_counts.iloc[0]["count"]} times)')

print(f' Error analysis saved to {ERR_DIR}/vit_confused_pairs.csv\n')

Most common confusion: "steak" misclassified as "filet_mignon" (26 times)
 Error analysis saved to /kaggle/working/results_vit/error_analysis/vit_confused_pairs.csv



In [11]:
# Packaging Results

zip_path = '/kaggle/working/food101_vit_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [OUT_DIR, CKPT_DIR]:
        for root, _, files in os.walk(folder):
            for file in files:
                fp = os.path.join(root, file)
                zf.write(fp, os.path.relpath(fp, '/kaggle/working'))

print(f' Results archive ready at {zip_path}')

from IPython.display import FileLink, display
display(FileLink('food101_vit_results.zip'))

 Results archive ready at /kaggle/working/food101_vit_results.zip


/kaggle/working/food101_vit_results.zip